In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [71]:
class MHA(nn.Module):
  def __init__(self, d_model, num_heads, dropout=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    assert self.head_dim * self.num_heads == self.d_model

    self.fc_q = nn.Linear(d_model, d_model)
    self.fc_k = nn.Linear(d_model, d_model)
    self.fc_v = nn.Linear(d_model, d_model)
    self.fc_o = nn.Linear(self.d_model, self.d_model)

    self.dropout = nn.Dropout(dropout)

  def forward(self, q, k, v, mask):
    batch, seq_len, _ = q.size()
    q = self.fc_q(q)
    k = self.fc_k(k)
    v = self.fc_v(v)

    q = q.view(batch, -1, self.num_heads, self.head_dim)
    k = k.view(batch, -1, self.num_heads, self.head_dim)
    v = v.view(batch, -1, self.num_heads, self.head_dim)

    scores =  torch.matmul(q, k.transpose(-2, -1))/ math.sqrt(self.head_dim)

    if mask is not None:
      scores = scores.masked_fill(mask == 0, -1e9)

    attn_weights = F.softmax(scores, dim=-1)
    attn_weights = self.dropout(attn_weights)

    out = torch.matmul(attn_weights, v)
    out = out.transpose(1,2).contiguous().view(batch, -1, self.d_model)
    out = self.fc_o(out)

    return out

In [72]:
class MQA(nn.Module):
  def __init__(self, d_model, num_heads, dropout=0.1):
    super().__init__()

    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    assert self.head_dim * self.num_heads == self.d_model

    self.fc_q = nn.Linear(d_model, d_model)
    self.fc_k = nn.Linear(d_model, self.head_dim)
    self.fc_v = nn.Linear(d_model, self.head_dim)

    self.fc_o = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(dropout)

  def forward(self, q, k, v, mask=None):
    batch, q_len, _ = q.size()
    _, kv_len, _ = k.size()
    q = self.fc_q(q)
    k = self.fc_k(k)
    v = self.fc_v(v)

    q = q.view(batch, q_len, self.num_heads, self.head_dim)
     # 形状: (batch_size, num_heads, seq_len, head_dim)
    q = q.transpose(1,2)

    k = k.unsqueeze(1).expand(batch, self.num_heads, kv_len, self.head_dim)
    v = v.unsqueeze(1).expand(batch, self.num_heads, kv_len, self.head_dim)

    scores = torch.matmul(q, k.transpose(-2,-1))/math.sqrt(self.head_dim)

    if mask is not None:
      scores = scores.masked_fill(mask==0, -1e9)

    attn_weights = F.softmax(scores, dim = -1)
    attn_weights = self.dropout(attn_weights)

    out = torch.matmul(attn_weights, v)

    out = out.transpose(1,2).contiguous().view(batch, q_len ,self.d_model)
    out = self.fc_o(out)
    return out



In [73]:
class GQA(nn.Module):
  def __init__(self, d_model, num_heads, num_groups, dropout=0.1):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.num_groups = num_groups  # KV组的数量
    self.head_dim = d_model // num_heads # 每个头的维度

    assert self.head_dim * self.num_heads == self.d_model
    assert self.num_heads % self.num_groups == 0 # self.num_groups 取整

    self.heads_per_group = self.num_heads // self.num_groups

    self.fc_q = nn.Linear(self.d_model, self.d_model)
    self.fc_k = nn.Linear(self.d_model, self.head_dim * self.num_groups)
    self.fc_v = nn.Linear(self.d_model, self.head_dim * self.num_groups)

    self.fc_o = nn.Linear(self.d_model, self.d_model)

    self.dropout = nn.Dropout(dropout)

  def forward(self, q, k, v, mask=None):
    batch, q_len, _ = q.size()
    _, kv_len, _ = k.size()

    q = self.fc_q(q)
    k = self.fc_k(k)
    v = self.fc_v(v)

    q = q.view(batch, q_len, self.num_heads, self.head_dim)
    k = k.view(batch, kv_len, self.num_groups, self.head_dim)
    v = v.view(batch, kv_len, self.num_groups, self.head_dim)

    q = q.transpose(1,2) # (batch_size, num_heads, q_len, head_dim)
    k = k.transpose(1,2) # (batch_size, num_groups, kv_len, head_dim)
    v = v.transpose(1,2) # (batch_size, num_groups, kv_len, head_dim)

    out = torch.zeros(batch, self.num_heads, q_len, self.head_dim, device=q.device)

    for i in range(self.num_groups):
      start_idx = i * self.heads_per_group
      end_idx = (i + 1) * self.heads_per_group
      group_q = q[:, start_idx:end_idx]

      group_k = k[:, i:i+1].expand(-1, self.heads_per_group, -1, -1)
      group_v = v[:, i:i+1].expand(-1, self.heads_per_group, -1, -1)

      scores = torch.matmul(group_q, group_k.transpose(-2,-1))/math.sqrt(self.head_dim)

      if mask is not None:
        scores = scores.masked_fill(mask==0, -1e9)

      attn_weights = F.softmax(scores, dim = -1)
      attn_weights = self.dropout(attn_weights)

      out[:, start_idx:end_idx] = torch.matmul(attn_weights, group_v)
    out = out.transpose(1,2).contiguous().view(batch, q_len, self.d_model)
    out = self.fc_o(out)
    return out



In [74]:
def test_attention_mechanisms():
  batch_size = 4
  seq_len = 10
  d_model = 512
  num_heads = 16
  num_groups = 4

  x = torch.randn(batch_size, seq_len, d_model)

  mha = MHA(d_model, num_heads)
  mqa = MQA(d_model, num_heads)
  gqa = GQA(d_model, num_heads, num_groups)

  mha_output = mha(x,x,x,None)
  mqa_output = mqa(x,x,x,None)
  gqa_output = gqa(x,x,x,None)

  print(f"MHA output shape:{mha_output.shape}")
  print(f"MQA output shape:{mqa_output.shape}")
  print(f"GQA output shape:{gqa_output.shape}")

  mha_params = sum(p.numel() for p in mha.parameters())
  mqa_params = sum(p.numel() for p in mqa.parameters())
  gqa_params = sum(p.numel() for p in gqa.parameters())
  print(f"MHA 参数数量: {mha_params}")
  print(f"MQA 参数数量: {mqa_params}")
  print(f"GQA 参数数量: {gqa_params}")
  print(f"MQA 相对于 MHA 的参数减少: {1 - mqa_params/mha_params:.2%}")
  print(f"GQA 相对于 MHA 的参数减少: {1 - gqa_params/mha_params:.2%}")



if __name__ == "__main__":
  test_attention_mechanisms()

MHA output shape:torch.Size([4, 10, 512])
MQA output shape:torch.Size([4, 10, 512])
GQA output shape:torch.Size([4, 10, 512])
MHA 参数数量: 1050624
MQA 参数数量: 558144
GQA 参数数量: 656640
MQA 相对于 MHA 的参数减少: 46.88%
GQA 相对于 MHA 的参数减少: 37.50%
